# Mechanistic Explanation with Gemma4EC

> **Predict an EC number from a reaction SMILES, then let the model explain its own reasoning, grounded in its own attention weights and enzyme data.**

[![Model](https://img.shields.io/badge/🤗%20HuggingFace-Gemma4EC--9B--Predict-blue)](https://huggingface.co/PlanesLab/Gemma4EC-9B-Predict)
[![Chat](https://img.shields.io/badge/🤗%20HuggingFace-TxGemma--9B--Chat-blue)](https://huggingface.co/google/txgemma-9b-chat)
[![BRENDA](https://img.shields.io/badge/Database-BRENDA_Enzymes-orange)](https://www.brenda-enzymes.org)

---

## What this notebook does

Three components work in sequence:

1. **[Gemma4EC-9B-Predict](https://huggingface.co/PlanesLab/Gemma4EC-9B-Predict)** takes a reaction SMILES and predicts its three-level EC number, while also returning layer-wise attention tensors.
2. **[BertViz](https://github.com/jessevig/bertviz)** renders an interactive attention heatmap over the SMILES tokens; we also extract a normalised per-token importance score.
3. **[TxGemma-9B-Chat](https://huggingface.co/google/txgemma-9b-chat)** receives the reaction, the predicted EC, the attention profile, and a BRENDA sub-subclass summary, then builds a step-by-step catalytic mechanism across four conversational turns.

### Pipeline at a glance

| Step | What happens | Key output |
|:----:|-------------|------------|
| 1 | Load a reaction SMILES  | `RXN_SMILES` |
| 2 | Load Gemma4EC-9B-Predict | predict model + LoRA adapter |
| 3 | Run inference; capture all-layer attention tensors | `PREDICTED_EC`, `ATTENTION` |
| 4 | BertViz interactive attention map  | heatmap widget |
| 5 | Normalise attention → \[0, 1\] per SMILES token | `ATTN_NORM_1D` |
| 6 | BRENDA lookup for the predicted EC sub-subclass | `BRENDA_INFO` |
| 7 | Load TxGemma-9B-Chat | freed VRAM |
| 8 | 4-turn mechanistic dialogue | printed + rendered explanation |

### Expected outputs

By the end of this notebook you will have:
- An **interactive attention heatmap** (BertViz widget) over the reaction SMILES.
- A **ranked token table** showing which SMILES substructures drove the EC prediction most strongly.
- A **detailed, literature-grounded mechanistic explanation** connecting model attention to enzyme catalysis.

## Setup & Requirements

### 1. Clone and install

```bash
git clone https://github.com/PlanesLab/Gemma4EC.git
cd Gemma4EC
conda env create -f envs/gemma4ec_env.yaml
conda activate gemma4ec
```

Install BertViz (not bundled in the main environment — required for the attention widget):

```bash
pip install bertviz
```


---

### 2. GPU requirements

Both models are loaded **sequentially** — only one is in GPU memory at a time.

| Quantisation | Min VRAM | How to set |
|---|:---:|---|
| 4-bit NF4 *(default)* | ~12 GB | `LOAD_IN_4BIT = True` |
| `bfloat16` | ~20 GB | `LOAD_IN_4BIT = False` |

---

### 3. Hugging Face access token

`google/txgemma-9b-predict` and `google/txgemma-9b-chat` are **gated models**. You must accept their licence on the Hub before they can be downloaded.

**One-time setup:**

1. Create a free account at [huggingface.co](https://huggingface.co).
2. Accept the model licences:
   - [google/txgemma-9b-predict](https://huggingface.co/google/txgemma-9b-predict)
   - [google/txgemma-9b-chat](https://huggingface.co/google/txgemma-9b-chat)
3. Generate a **read** token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).
4. Export it before launching Jupyter …

   ```bash
   export HUGGINGFACE_TOKEN=hf_...
   ```

   … or paste it into the interactive prompt in the cell below.

> **Tip:** Setting `HUGGINGFACE_TOKEN` as a shell variable means you never need to paste credentials into the notebook itself.

---

## Step 1 — Hugging Face authentication

Set your token **before launching Jupyter** so it is available as an environment variable:

```bash
export HUGGINGFACE_TOKEN=hf_...
bash launch_jupyter_gpu.sh
```

The cell below picks it up automatically — nothing is written to disk.

In [ ]:
import os

# huggingface_hub reads HF_TOKEN automatically — no login() call needed,
# which avoids writing to ~/.cache/huggingface/ (home dir disk quota issue).
_hf_token = os.environ.get("HUGGINGFACE_TOKEN", "") or os.environ.get("HF_TOKEN", "")
if _hf_token:
    os.environ["HF_TOKEN"] = _hf_token
    print("HF_TOKEN set from environment variable — authentication ready.")
else:
    print("WARNING: No HF token found in environment.")
    print("  Set it before launching Jupyter:  export HUGGINGFACE_TOKEN=hf_...")
    print("  Gated models (txgemma-9b-predict, txgemma-9b-chat) will fail to download.")

HF_TOKEN set from environment variable — authentication ready.


---

## Step 2 — Configure paths & hyperparameters

All paths are resolved automatically from the repo root — nothing to change for a standard clone.

The LoRA adapter (`ADAPTER_PATH`) is downloaded from the Hugging Face Hub on first run (~300 MB) and cached automatically.
All other variables are safe to leave at their defaults.

In [ ]:
import os
from pathlib import Path

# ── Resolve repo root ─────────────────────────────────────────────────────────
def _find_repo_root(start: Path) -> Path:
    for candidate in [start] + list(start.parents):
        if (candidate / "prompts" / "prompts.json").exists():
            return candidate
        if (candidate / "Gemma4EC" / "prompts" / "prompts.json").exists():
            return candidate / "Gemma4EC"
    raise FileNotFoundError(
        f"Cannot find prompts/prompts.json searching up from {start}.\n"
        "Make sure you are inside (or above) the Gemma4EC repo directory."
    )

_HERE = Path(".").resolve()
_REPO = _find_repo_root(_HERE)

# ── HF cache ─────────────────────────────────────────────────────────────────
_HF_CACHE = Path(os.environ.get("HF_HOME", str(_REPO / ".cache/gemma4ec/hf")))
_HF_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(_HF_CACHE)

# ── Model IDs ────────────────────────────────────────────────────────────────
BASE_PREDICT = "google/txgemma-9b-predict"
BASE_CHAT    = "google/txgemma-9b-chat"
ADAPTER_PATH = "PlanesLab/Gemma4EC-9B-Predict"

TEST_CSV     = str(_REPO / "data/ECReact/test.csv")
PROMPTS_JSON = str(_REPO / "prompts/prompts.json")
BRENDA_INDEX = str(_REPO / "chat/data/brenda_index.json")

# ── Hyperparameters ───────────────────────────────────────────────────── ✏️ ──
TEMPLATE_KEY     = "EC_NUMBER_PROMPT_2"
ANSWER_MARKER    = "Answer:"
LOAD_IN_4BIT     = True   # set False if GPU VRAM >= 20 GB
SHORT_SMILES_MAX = 300
RANDOM_SEED      = 42
MAX_TOKENS       = 400
N_LAYERS_VIZ     = 6

# ── Validate all required files ───────────────────────────────────────────────
_required = {"test.csv": TEST_CSV, "prompts.json": PROMPTS_JSON, "brenda_index.json": BRENDA_INDEX}
_missing  = [name for name, p in _required.items() if not Path(p).exists()]

print(f"Repo root : {_REPO}")
print(f"HF cache  : {_HF_CACHE}")
if _missing:
    print(f"\n⚠  Missing files — notebook will fail at later steps: {_missing}")
else:
    print("  All required data files found ✓")

In [ ]:
import json
import random
import pandas as pd

# ── auto-select a short reaction from the test set ────────────────────────────
df      = pd.read_csv(TEST_CSV)
df_short = df[df["rxn_smiles"].str.len() <= SHORT_SMILES_MAX].reset_index(drop=True)
_rng    = random.Random(RANDOM_SEED)
_row    = df_short.iloc[_rng.randint(0, len(df_short) - 1)]

print(f"Auto-selected reaction  (SMILES len = {len(_row['rxn_smiles'])} chars):")
print(f"  SMILES  : {_row['rxn_smiles']}")
print(f"  True EC : {_row['ec']}")
print()

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  ✏️  USER INPUT                                                              ║
# ║  Paste your own reaction SMILES below (as a plain string).                  ║
# ║  Leave RXN_SMILES = None to use the auto-selected example above.            ║
# ║  TRUE_EC is optional — set it to the known EC for comparison, or leave None.║
# ╚══════════════════════════════════════════════════════════════════════════════╝

RXN_SMILES = None   # e.g. "CC(=O)O.NC>>CC(=O)N.O"
TRUE_EC    = None   # e.g. "6.3.1.1"

# ── resolve: fall back to auto-selected if user left None ─────────────────────
if RXN_SMILES is None:
    RXN_SMILES = _row["rxn_smiles"]
    TRUE_EC    = str(_row["ec"]) if TRUE_EC is None else TRUE_EC
elif TRUE_EC is None:
    TRUE_EC = "unknown"

with open(PROMPTS_JSON) as f:
    PREDICT_TEMPLATE = json.load(f)[TEMPLATE_KEY]

PROMPT = PREDICT_TEMPLATE.format(reaction_smiles=RXN_SMILES)

print("Using reaction:")
print(f"  SMILES  : {RXN_SMILES}")
print(f"  True EC : {TRUE_EC}")
print(f"  Prompt  :\n{PROMPT}")


Auto-selected reaction  (SMILES len = 245 chars):
  SMILES  : NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)([O-])OP(=O)([O-])OC[C@H]3O[C@@H](n4cnc5c(N)ncnc54)[C@H](OP(=O)([O-])[O-])[C@@H]3O)[C@@H](O)[C@H]2O)c1.O=C([O-])CCC[C@H](O)/C=C\C=C\C=C\[C@H](O)C/C=C\CCCCCO>>O=C([O-])CCC[C@H](O)/C=C\C=C\C=C\C(=O)C/C=C\CCCCCO
  True EC : 1.03.01

Using reaction:
  SMILES  : NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)([O-])OP(=O)([O-])OC[C@H]3O[C@@H](n4cnc5c(N)ncnc54)[C@H](OP(=O)([O-])[O-])[C@@H]3O)[C@@H](O)[C@H]2O)c1.O=C([O-])CCC[C@H](O)/C=C\C=C\C=C\[C@H](O)C/C=C\CCCCCO>>O=C([O-])CCC[C@H](O)/C=C\C=C\C=C\C(=O)C/C=C\CCCCCO
  True EC : 1.03.01
  Prompt  :
Instructions: Given a reaction SMILES, predict the first three parts of its Enzyme Commission (EC) number (class.subclass.sub-subclass).
EC numbers are four-part hierarchical codes (e.g., 1.3.2.4) describing the type of enzyme and its reaction mechanism.
Only output the first three digits (e.g., 1.03.02) representing the enzyme's general class, subclass, and sub-

In [ ]:
import gc
import os
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

_predict_ready = (
    "predict_model" in vars() and isinstance(predict_model, PeftModel)
    and "tok_predict" in vars()
)

if _predict_ready:
    print("[predict] Already in memory — skipping reload. Restart kernel to force a fresh load.")
else:
    # Free GPU memory from any failed previous attempt
    for _var in ("predict_model", "tok_predict"):
        if _var in vars():
            del vars()[_var]
    gc.collect()
    torch.cuda.empty_cache()

    print("[predict] Loading tokenizer …")
    tok_predict = AutoTokenizer.from_pretrained(BASE_PREDICT)
    if tok_predict.pad_token is None:
        tok_predict.pad_token = tok_predict.eos_token

    if LOAD_IN_4BIT:
        quant = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16
        )
        print("[predict] Loading base model (4-bit NF4) …")
        predict_model = AutoModelForCausalLM.from_pretrained(
            BASE_PREDICT, quantization_config=quant,
            attn_implementation="eager", device_map="auto",
        )
    else:
        print("[predict] Loading base model (bfloat16) …")
        predict_model = AutoModelForCausalLM.from_pretrained(
            BASE_PREDICT, torch_dtype=torch.bfloat16,
            attn_implementation="eager", device_map="cuda:0",
        )

    print(f"[predict] Loading LoRA adapter: {ADAPTER_PATH}")
    if os.path.isdir(ADAPTER_PATH):
        _adapter_local = ADAPTER_PATH
    else:
        from huggingface_hub import snapshot_download
        _hf_hub_cache = os.path.join(os.environ.get("HF_HOME", str(Path.home() / ".cache/huggingface")), "hub")
        _adapter_local = snapshot_download(repo_id=ADAPTER_PATH, cache_dir=_hf_hub_cache)

    predict_model = PeftModel.from_pretrained(predict_model, _adapter_local, is_trainable=False)
    predict_model.eval()
    print("[predict] Ready.")

In [ ]:
import subprocess, shutil

# GPU memory
print("=== GPU ===")
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.free,memory.total",
                    "--format=csv,noheader"], capture_output=True, text=True)
print(r.stdout.strip() if r.returncode == 0 else "nvidia-smi not available")

# Disk space on /scratch
print("\n=== Disk (/scratch) ===")
total, used, free = shutil.disk_usage("/scratch")
print(f"  total: {total/1e9:.1f} GB   used: {used/1e9:.1f} GB   free: {free/1e9:.1f} GB")

# bitsandbytes
print("\n=== bitsandbytes ===")
try:
    import bitsandbytes as bnb
    print(f"  version: {bnb.__version__}")
    print(f"  CUDA available: {bnb.cuda_setup.CUDASetup.get_instance().cuda_available}")
except Exception as e:
    print(f"  ERROR: {e}")


In [ ]:
def parse_ec(text: str) -> str:
    tail = text.rsplit(ANSWER_MARKER, 1)[-1] if ANSWER_MARKER in text else text
    return tail.strip().splitlines()[0].strip() if tail.strip() else ""

@torch.no_grad()
def infer_with_attention(model, tok, prompt, max_tokens=400, max_new_tokens=8):
    device = next(model.parameters()).device
    enc = tok(prompt, return_tensors="pt", truncation=True, max_length=max_tokens).to(device)
    prompt_len = enc.input_ids.shape[1]

    gen_ids = model.generate(
        input_ids=enc.input_ids,
        attention_mask=enc.attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )
    full_text = tok.decode(gen_ids[0], skip_special_tokens=True)
    predicted_ec = parse_ec(full_text)

    # Forward pass over full sequence (prompt + generated tokens) for attentions
    full_mask = torch.ones_like(gen_ids)
    out = model(input_ids=gen_ids, attention_mask=full_mask, output_attentions=True)

    # Keep batch dim [1, H, T, T] per layer — needed for BertViz
    attention = [a.cpu().float() for a in out.attentions]
    tokens    = tok.convert_ids_to_tokens(gen_ids[0].tolist())

    return predicted_ec, attention, tokens, prompt_len


PREDICTED_EC, ATTENTION, TOKENS, PROMPT_LEN = infer_with_attention(
    predict_model, tok_predict, PROMPT, MAX_TOKENS
)

print(f"True EC       : {TRUE_EC}")
print(f"Predicted EC  : {PREDICTED_EC}")
print(f"Tokens total  : {len(TOKENS)}  (prompt={PROMPT_LEN}, generated={len(TOKENS)-PROMPT_LEN})")


In [ ]:
def smiles_token_span(prompt: str, tok, max_tokens: int):
    """Return (start, end) token indices of the SMILES section in the prompt."""
    try:
        enc = tok(prompt, truncation=True, max_length=max_tokens, return_offsets_mapping=True)
    except Exception:
        return None, None

    offsets   = enc["offset_mapping"]
    sm_marker = "Reaction SMILES:"
    ans_marker = "\nAnswer:"

    sm_pos = prompt.find(sm_marker)
    if sm_pos == -1:
        return None, None
    char_start = sm_pos + len(sm_marker) + 1      # skip the space after ':'

    ans_pos  = prompt.find(ans_marker)
    char_end = ans_pos if ans_pos != -1 else len(prompt)

    tok_start = next((i for i, (s, e) in enumerate(offsets) if e > char_start), None)
    tok_end   = next((i for i, (s, e) in enumerate(offsets) if s >= char_end), len(offsets))

    return tok_start, tok_end


SM_START, SM_END = smiles_token_span(PROMPT, tok_predict, MAX_TOKENS)
print(f"SMILES token span : [{SM_START}, {SM_END})  ({SM_END - SM_START} tokens)")
print(f"Sample SMILES tokens : {TOKENS[SM_START:SM_START+10]} …")


In [ ]:
# ── BertViz interactive head_view ────────────────────────────────────────────
# Shows only the SMILES tokens + generated EC tokens from the last N layers.
# Use the layer/head selectors in the widget to explore.

from bertviz import head_view

tok_s = SM_START if SM_START is not None else 0
tok_e = len(TOKENS)                              # includes EC generation tokens

attn_viz   = [a[:, :, tok_s:tok_e, tok_s:tok_e] for a in ATTENTION[-N_LAYERS_VIZ:]]
tokens_viz = TOKENS[tok_s:tok_e]

print(f"BertViz: showing tokens [{tok_s}:{tok_e}] — {len(tokens_viz)} tokens "
      f"(last {N_LAYERS_VIZ} layers)")
print(f"Predicted EC: {PREDICTED_EC}  |  True EC: {TRUE_EC}")
print()

head_view(attn_viz, tokens_viz)


In [ ]:
import numpy as np

# ── raw attention: average over all layers & heads, last-token query ──────────
stacked = torch.stack(ATTENTION)            # [n_layers, 1, n_heads, seq, seq]
avg_attn = stacked[:, 0, :, -1, :].mean(dim=(0, 1)).numpy()   # [seq]

# Slice to SMILES region
smiles_attn_raw  = avg_attn[SM_START:SM_END]
SMILES_TOKENS    = TOKENS[SM_START:SM_END]

# ── min-max normalise → [0, 1] ────────────────────────────────────────────────
v_min, v_max = smiles_attn_raw.min(), smiles_attn_raw.max()
attn_norm = (smiles_attn_raw - v_min) / (v_max - v_min + 1e-9)

# Round to 1 decimal place as requested
ATTN_NORM_1D = np.round(attn_norm, 1)

print(f"Attention vector normalised — {len(ATTN_NORM_1D)} SMILES tokens")
print(f"  min (raw): {v_min:.4e}   max (raw): {v_max:.4e}")
print(f"  After normalisation range: [{ATTN_NORM_1D.min():.1f}, {ATTN_NORM_1D.max():.1f}]")


In [ ]:
import pandas as pd
from IPython.display import display

# Clean token labels (strip SentencePiece artefacts)
def clean_tok(t: str) -> str:
    return t.replace("\u2581", " ").replace("\u0120", " ").replace("\u010a", "\\n").strip()

attn_df = pd.DataFrame({
    "token_idx" : list(range(SM_START, SM_END)),
    "token"     : [clean_tok(t) for t in SMILES_TOKENS],
    "attn_norm" : ATTN_NORM_1D,
})

# Highlight the top-10 tokens (highest attention)
top10_idx = attn_df["attn_norm"].nlargest(10).index

def highlight_top(row):
    color = "background-color: #ffcc80" if row.name in top10_idx else ""
    return [color] * len(row)

print("Attention profile — SMILES tokens (orange = top-10 most attended):")
display(attn_df.style.apply(highlight_top, axis=1).format({"attn_norm": "{:.1f}"}))


In [ ]:
import json

def ec_to_ec3(ec_str: str) -> str | None:
    parts = ec_str.strip().split(".")
    if len(parts) < 3:
        return None
    try:
        return ".".join(str(int(p)) for p in parts[:3])
    except ValueError:
        return None

if "BRENDA_DICT" not in vars():
    with open(BRENDA_INDEX, encoding="utf-8") as f:
        BRENDA_DICT = json.load(f)
    print(f"BRENDA index loaded ({len(BRENDA_DICT)} entries).")
else:
    print("BRENDA index already loaded — skipping.")

EC3 = ec_to_ec3(PREDICTED_EC)
BRENDA_INFO = BRENDA_DICT.get(EC3, f"No BRENDA entry found for EC {EC3}.")

print(f"Predicted EC   : {PREDICTED_EC}")
print(f"EC sub-subclass: {EC3}")
print()
print("BRENDA entry (first 1000 chars):")
print(BRENDA_INFO[:1000])
if len(BRENDA_INFO) > 1000:
    print("… [truncated for display]")

In [ ]:
import gc
import subprocess

for _var in ("predict_model", "tok_predict"):
    if _var in vars():
        del vars()[_var]
gc.collect()
torch.cuda.empty_cache()

result = subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.used,memory.free", "--format=csv,noheader,nounits"],
    capture_output=True, text=True,
)
print("Predict model unloaded. GPU memory (used / free MiB):")
print(result.stdout.strip() if result.returncode == 0 else "nvidia-smi not available")

In [ ]:
_chat_ready = "chat_model" in vars() and "tok_chat" in vars()

if _chat_ready:
    print("[chat] Already in memory — skipping reload. Restart kernel to force a fresh load.")
else:
    # Free GPU memory from any failed previous attempt
    for _var in ("chat_model", "tok_chat"):
        if _var in vars():
            del vars()[_var]
    gc.collect()
    torch.cuda.empty_cache()

    print("[chat] Loading tokenizer …")
    tok_chat = AutoTokenizer.from_pretrained(BASE_CHAT)
    if tok_chat.pad_token is None:
        tok_chat.pad_token = tok_chat.eos_token

    if LOAD_IN_4BIT:
        quant_chat = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16
        )
        print("[chat] Loading TxGemma-9B-Chat (4-bit NF4) …")
        chat_model = AutoModelForCausalLM.from_pretrained(
            BASE_CHAT, quantization_config=quant_chat, device_map="auto",
        )
    else:
        print("[chat] Loading TxGemma-9B-Chat (bfloat16) …")
        chat_model = AutoModelForCausalLM.from_pretrained(
            BASE_CHAT, torch_dtype=torch.bfloat16, device_map="cuda:0",
        )

    chat_model.eval()
    print("[chat] Ready.")

---

## Step 11 — Iterative mechanistic explanation (4-turn dialogue)

The conversation builds context progressively — each turn adds one layer of evidence before the final synthesis.

| Turn | Context added | Goal |
|:----:|--------------|------|
| **1** | Reaction SMILES + predicted EC | Identify functional groups, bonds formed/broken, and the broad transformation type |
| **2** | Normalised per-token attention profile | Determine which SMILES substructures the model focused on and whether they are chemically relevant |
| **3** | BRENDA sub-subclass entry | Generate a detailed step-by-step mechanism with cofactors and intermediate species |
| **4** | *(synthesis)* | Write a concise paragraph connecting model attention highlights to the catalytic mechanism |

The `chat()` helper below maintains a running `history` list that is re-encoded on every call, giving the model full conversational context at each turn.

In [ ]:
from IPython.display import Markdown, display as ipy_display

history = []   # list of {"role": "user"|"model", "content": str}


def chat(user_msg: str, max_new_tokens: int = 512) -> str:
    """Send a turn to TxGemma-9B-Chat and return the response string."""
    history.append({"role": "user", "content": user_msg})
    prompt_str = tok_chat.apply_chat_template(
        history, tokenize=False, add_generation_prompt=True
    )
    inputs = tok_chat(prompt_str, return_tensors="pt").to(chat_model.device)

    with torch.no_grad():
        out = chat_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )

    response = tok_chat.decode(
        out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
    )
    history.append({"role": "model", "content": response})
    return response


def show(turn: int, role: str, text: str):
    header = f"### Turn {turn} — {role}"
    ipy_display(Markdown(f"{header}\n\n{text}"))
    print()


In [ ]:
# ── Build attention profile string ──────────────────────────────────────────
def attn_profile_str(tokens, weights, n_top=15) -> str:
    """Return a compact token:weight string, showing only the n_top highest."""
    pairs = sorted(zip(tokens, weights), key=lambda x: -x[1])
    lines = []
    for t, w in pairs[:n_top]:
        label = t.replace("▁", "").replace("Ġ", "").strip() or repr(t)
        lines.append(f"{label}:{w:.1f}")
    return "  ".join(lines)

clean_tokens = [clean_tok(t) for t in SMILES_TOKENS]
ATTN_STR     = attn_profile_str(clean_tokens, ATTN_NORM_1D)

# ── Turn 1 prompt ─────────────────────────────────────────────────────────────
TURN1 = f"""You are an expert enzymologist and mechanistic chemist.
I will give you information about an enzyme-catalysed reaction in several steps.
Let's build up to a complete mechanistic explanation together.

Here is a reaction SMILES and the EC number predicted by Gemma4EC \
(a TxGemma-9B model fine-tuned on the raw ECReact dataset):

Reaction SMILES:
{RXN_SMILES}

Predicted EC number: {PREDICTED_EC}

Based on the SMILES alone, identify:
1. Which functional groups are present in the substrate(s) and product(s).
2. What bonds are formed or broken.
3. What broad type of chemical transformation this represents \
(e.g. oxidation/reduction, group transfer, hydrolysis, isomerisation, etc.).

Keep the answer focused — we will expand in later turns."""

r1 = chat(TURN1, max_new_tokens=400)
show(1, "Reaction context + EC prediction", TURN1)
show(1, "Gemma-9B-Chat", r1)

In [ ]:
# ── build a pretty token-weight table for the prompt ─────────────────────────
def token_weight_table(tokens, weights) -> str:
    rows = ["Token | Norm. attention (0–1)", "-----|--------------------"]
    for t, w in zip(tokens, weights):
        label = t.replace("\u2581", "").replace("\u0120", "").strip() or repr(t)
        bar = "█" * int(w * 10)
        rows.append(f"`{label}` | {w:.1f}  {bar}")
    return "\n".join(rows)

ATTN_TABLE = token_weight_table(clean_tokens, ATTN_NORM_1D)

TURN2 = f"""The model's attention weights reveal which SMILES tokens it focused on most when \
predicting EC {PREDICTED_EC}.  Below is the normalised attention profile \
(0.0 = ignored, 1.0 = maximum attention) over the SMILES tokens:

{ATTN_TABLE}

Given this attention profile:
1. Which atoms or functional groups did the model attend to most strongly?
2. Are these the chemically relevant sites for a EC {PREDICTED_EC} reaction? Why or why not?
3. Does the attention pattern suggest the model understood the reaction correctly?"""

r2 = chat(TURN2, max_new_tokens=400)
show(2, "Attention profile", TURN2)
show(2, "Gemma-9B-Chat", r2)


In [ ]:
TURN3 = f"""Here is the BRENDA enzyme database information for EC sub-subclass {EC3}:

{BRENDA_INFO}

Using everything we have discussed plus this BRENDA reference, provide a \
detailed step-by-step mechanistic explanation for this reaction:

1. Name the type of catalysis (acid-base, covalent, metal-ion assisted, etc.).
2. Describe the key catalytic steps, including any intermediate species, \
   cofactors, or coenzymes involved.
3. Explain how the structural features of the reactant(s) and product(s) \
   support the EC {PREDICTED_EC} classification.
4. Note which atoms/bonds undergo the most significant changes and why \
   those relate to the EC sub-subclass {EC3}."""

r3 = chat(TURN3, max_new_tokens=600)
show(3, "BRENDA sub-subclass info + full mechanism request", TURN3)
show(3, "Gemma-9B-Chat", r3)


In [ ]:
TURN4 = f"""Finally, write a concise synthesis paragraph (5–8 sentences) that connects:
- The structural features of the SMILES that led to the EC {PREDICTED_EC} prediction,
- The specific SMILES tokens that received the highest attention \
  (according to the attention profile above),
- The known catalytic mechanism for EC {EC3} reactions.

The paragraph should explain *why* Gemma4EC correctly predicted EC {PREDICTED_EC} \
for this reaction, framing the model's attention as mechanistic insight."""

r4 = chat(TURN4, max_new_tokens=400)
show(4, "Synthesis: attention → mechanism", TURN4)
show(4, "Gemma-9B-Chat", r4)


In [ ]:
# ── Print the complete conversation as plain text ────────────────────────────
print("=" * 80)
print("FULL MECHANISTIC EXPLANATION CONVERSATION")
print(f"Reaction SMILES : {RXN_SMILES[:80]}...")
print(f"True EC         : {TRUE_EC}")
print(f"Predicted EC    : {PREDICTED_EC}")
print("=" * 80)
for i, turn in enumerate(history):
    role = "USER" if turn["role"] == "user" else "GEMMA-CHAT"
    print(f"\n[Turn {i//2 + 1} — {role}]")
    print(turn["content"])
    print("-" * 40)
